# TP 1 · Mettre en place un chatbot RAG
**Module FM02 · IA Générative · M2 Lead Developer Full Stack · ENSITECH**

**Objectif** : construire un pipeline RAG complet sur le corpus fourni (documentation interne fictive de l'entreprise Orbitale), avec une stack open source : sentence-transformers, ChromaDB, Ollama et Gradio.

| Étape | Consigne de la slide TP 1 | Fichier(s) à compléter |
|---|---|---|
| 1 | Charger le corpus | `src/rag/loaders.py` |
| 2 | Implémenter un chunking **justifié** | `src/rag/chunking.py`, `config/tp1.yaml` |
| 3 | Vectoriser (sentence-transformers) et indexer (ChromaDB) | `src/rag/embeddings.py`, `src/rag/vectorstore.py` |
| 4 | Connecter la recherche au LLM (Ollama, Gemini en secours) | `src/rag/prompts.py`, `src/rag/llm.py`, `src/rag/pipeline.py` |
| 5 | Interface Gradio | `app/app_gradio.py` |

**Livrable** (dépôt GitHub) : README (avec la justification du chunking), code source, `reports/TP1_exemples_QR.md` avec 5 questions-réponses testées.

> **Avant de commencer** : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, le TP fonctionne avec l'API Gemini (étape 0.5).


> **Méthode de travail** : les fonctions à écrire contiennent `raise NotImplementedError("TODO ...")`. Modifiez les fichiers de `src/` (panneau Fichiers de Colab, double-clic pour éditer, ou dans votre éditeur local puis `git pull`). La commande `pytest` (étape 6) indique ce qui reste à faire.

## Étape 0 · Installation (à relancer à chaque nouvelle session Colab)

In [ ]:
# 0.1 GPU disponible ?
!nvidia-smi -L || echo "Pas de GPU : passez en T4 (Exécution > Modifier le type d'exécution) ou utilisez Gemini (0.5)" 

In [ ]:
# 0.2 Récupérer le projet (votre fork GitHub). Alternative : importer le zip puis !unzip
import os
REPO_URL = "https://github.com/VOTRE-COMPTE/fm02-tp-rag.git"   # à remplacer
REPO_DIR = "fm02-tp-rag"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
if IN_COLAB:
    %cd {REPO_DIR}
elif os.path.basename(os.getcwd()) == "notebooks":     # Jupyter en local : se placer à la racine du projet
    os.chdir("..")
print(os.getcwd())

In [ ]:
# 0.3 Dépendances (2 à 3 minutes)
!pip install -q -r requirements.txt

In [ ]:
# 0.4 Ollama dans Colab (GPU T4 recommandé) : installation, serveur en tâche de fond, modèle
OLLAMA_MODEL = "qwen3:8b"          # environ 5 Go ; sans GPU : "qwen3:1.7b" ou passer à Gemini
import subprocess, time, urllib.request
if IN_COLAB:
    !sudo apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434"); print("Ollama prêt"); break
    except Exception:
        time.sleep(1)
!ollama pull {OLLAMA_MODEL}

In [ ]:
# 0.5 Clés d'API facultatives, stockées dans les secrets Colab (icône clé à gauche) :
#   GEMINI_API_KEY (secours si pas de GPU, clé gratuite sur aistudio.google.com)
#   LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_BASE_URL (traces, compte gratuit sur cloud.langfuse.com)
# En local : fichier .env (voir .env.example). Ne jamais écrire une clé en clair dans le notebook.
import os
try:
    from google.colab import userdata
    for k in ["GEMINI_API_KEY", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
print({k: ("ok" if os.environ.get(k) else "absente") for k in ["GEMINI_API_KEY", "LANGFUSE_PUBLIC_KEY"]})

In [ ]:
# 0.6 Rendre le code de src/ importable, et le recharger automatiquement après chaque modification
import sys
sys.path[:0] = ["src", "."]
%load_ext autoreload
%autoreload 2

## Étape 1 · Ingestion et parsing
Le corpus mélange volontairement 4 formats : Markdown, texte, **PDF** (avec en-tête et pied de page répétés) et **HTML** (avec menu et pied de page). Objectif : un texte propre par document, avec ses métadonnées.

In [ ]:
from rag.loaders import load_corpus

docs = load_corpus("data/corpus")
for d in docs:
    print(f"{d.metadata['source']:32} {d.metadata['type']:5} page={d.metadata.get('page', '-')}  {len(d.text):5} caractères")

In [ ]:
# Vérifier le nettoyage : la page 2 du PDF ne doit plus contenir l'en-tête « Usage interne »
pdf_pages = [d for d in docs if d.metadata["type"] == "pdf"]
print(pdf_pages[1].text[:500])
print("\n--- HTML ---\n")
print([d for d in docs if d.metadata["type"] == "html"][0].text[:500])

> **Pour aller plus loin** : comparez l'extraction du PDF avec `pymupdf4llm` (`pip install pymupdf4llm`, puis `pymupdf4llm.to_markdown(path)`) : le tableau des plafonds est-il mieux conservé ? Quel effet sur la question « plafond d'une nuit d'hôtel à Paris » ?

## Étape 2 · Chunking
Deux stratégies : `fixed` (taille fixe + chevauchement) et `markdown` (par sections, puis taille fixe). Comparez les statistiques et **lisez des chunks** avant de choisir.

In [ ]:
from rag.chunking import chunk_documents, chunk_stats
from rag.config import RAGConfig

for strategy in ["fixed", "markdown"]:
    for size in [200, 500, 1000]:
        cfg = RAGConfig(chunking=strategy, chunk_size=size, chunk_overlap=size // 10)
        print(f"{strategy:9} {size:5} -> {chunk_stats(chunk_documents(docs, cfg))}")

In [ ]:
# Lire quelques chunks : une règle est-elle coupée en deux ? le titre de section est-il conservé ?
cfg = RAGConfig(chunking="markdown", chunk_size=500, chunk_overlap=60)
for c in chunk_documents(docs, cfg)[3:7]:
    print(c.id, "|", c.metadata.get("section"))
    print(c.text, "\n" + "-" * 80)

### TODO · Justification du chunking
Choisissez la stratégie, la taille et le chevauchement dans `config/tp1.yaml`, puis justifiez ce choix ici **et** dans le README (3 à 5 arguments, chiffres à l'appui : nombre de chunks, taille médiane, exemples de chunks bien ou mal coupés).

> **Pour aller plus loin** : essayez `RecursiveCharacterTextSplitter` de LangChain (`pip install langchain-text-splitters`) sur le même corpus, ou un chunking en **tokens** avec le tokenizer du modèle d'embedding. Les chunks diffèrent-ils beaucoup des vôtres ?

## Étape 3 · Embeddings et indexation dans ChromaDB
Le modèle par défaut est **BAAI/bge-m3** (multilingue). Règle d'or : le même modèle pour indexer et pour interroger.

In [ ]:
from rag.embeddings import Embedder
import numpy as np

embedder = Embedder("BAAI/bge-m3")      # téléchargement ~2 Go au premier lancement
print("device :", embedder.device, "| dimension :", embedder.dimension)

phrases = ["Combien de jours de congés ai-je par an ?",
           "Chaque salarié acquiert 25 jours ouvrés de congés payés.",
           "Le mot de passe doit compter 14 caractères."]
v = embedder.embed_documents(phrases)
print("similarité question / congés :", round(float(v[0] @ v[1]), 3))
print("similarité question / mot de passe :", round(float(v[0] @ v[2]), 3))

In [ ]:
from rag.config import load_config
from rag.pipeline import RAGPipeline

cfg = load_config("config/tp1.yaml")
pipe = RAGPipeline(cfg, embedder=embedder)
pipe.index("data/corpus", reset=True)

In [ ]:
for c in pipe.retrieve("Dans quel délai transmettre un arrêt maladie ?"):
    print(f"{c.rank}. {c.score:.3f}  {c.metadata['source']}  {c.text[:90]!r}")

> **Pour aller plus loin** : remplacez bge-m3 par `intfloat/multilingual-e5-small` (5 fois plus léger) puis par `sentence-transformers/all-MiniLM-L6-v2` (anglais). Comparez les scores de l'étape ci-dessus : que se passe-t-il avec un modèle anglais sur un corpus français ?

## Étape 4 · Génération ancrée avec le LLM
Le prompt applique la **règle d'or** du cours : si le contexte ne contient pas la réponse, le modèle répond « Je ne sais pas d'après les documents fournis ».

In [ ]:
from rag.prompts import SYSTEM_PROMPT, build_user_prompt
print(SYSTEM_PROMPT)
print("\n" + build_user_prompt("Exemple ?", pipe.retrieve("congés payés")[:2])[:600])

In [ ]:
res = pipe.ask("Dans quel délai et avec quel formulaire transmettre un arrêt maladie ?")
print(res.answer)
print("\nsources :", res.source_files, "| modèle :", res.model)
print(f"recherche {res.retrieval_s:.2f} s · génération {res.generation_s:.2f} s · {res.prompt_tokens} tokens en entrée")

In [ ]:
# Question hors corpus : le modèle doit refuser de répondre
print(pipe.ask("Quelle mutuelle santé Orbitale propose-t-elle ?").answer)

> **Pour aller plus loin** : comparez `qwen3:8b` avec `llama3.1:8b` ou `mistral:7b` (`!ollama pull ...` puis `cfg.ollama_model = ...`). Lequel respecte le mieux la règle d'or et la citation des sources ?

## Étape 5 · Cinq questions-réponses testées (livrable)
Choisissez 5 questions variées (une par type au moins : factuelle, tableau, PDF, HTML, hors corpus). Le rapport est écrit dans `reports/TP1_exemples_QR.md`.

In [ ]:
questions = [
    "Combien de jours de congés payés acquiert un collaborateur à temps plein ?",
    # TODO : ajouter 4 questions variées (tableau, PDF, HTML, hors corpus)
]

In [ ]:
from pathlib import Path

lines = ["# TP 1 · Exemples de questions-réponses", "",
         f"Configuration : `{cfg.name}` · chunking {cfg.chunking} {cfg.chunk_size} car. · top-k {cfg.top_k} · modèle {pipe.llm.__class__.__name__}", ""]
for i, q in enumerate(questions, 1):
    r = pipe.ask(q)
    lines += [f"## Q{i}. {q}", "", r.answer, "",
              "Sources : " + ", ".join(f"`{s}`" for s in r.source_files),
              f"Latence : {r.retrieval_s + r.generation_s:.1f} s", "",
              "Verdict (à compléter) : correcte / partielle / fausse, et pourquoi.", ""]
Path("reports/TP1_exemples_QR.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

## Étape 6 · Interface Gradio et tests
`share=True` crée une URL publique temporaire (72 h) : pratique pour montrer le chatbot.

In [ ]:
from app.app_gradio import build_app
demo = build_app(pipe)
demo.launch(share=True, debug=False)

In [ ]:
# Tests unitaires du TP 1 (sans GPU, avec un faux modèle) : tout doit passer avant le rendu
!python -m pytest -q -m 'not tp2 and not bonus'

> **Pour aller plus loin** : ajoutez une mémoire de conversation (reformuler « et pour les RTT ? » à partir de l'historique avant la recherche), ou le streaming de la réponse dans Gradio.

## Étape 7 · Rendu sur GitHub
Depuis Colab : *Fichier > Enregistrer une copie dans GitHub* pour le notebook, et pour le code :
```bash
git config user.email "vous@exemple.com"; git config user.name "Prénom Nom"
git add -A && git commit -m "TP1 : pipeline RAG complet" && git push
```
(un jeton GitHub est nécessaire pour `git push` depuis Colab). Ne commitez **jamais** de clé d'API : elles restent dans les secrets Colab ou le fichier `.env` (ignoré par Git).